<div style="text-align:center;">
  <h1 size=10>
    <b>MLOps PROJECT</b><br>
    <b></b>
  </h1>
</div>

<h2 style="text-align:center;">
Master's in Data Science and Advanced Analytics - NOVA IMS (25/26)
</h2>

**Group ??**
- Bárbara Franco (20250388)
- Inês Caetano (20221950)
- João Bernardo (20221889)
- Maria Carvalho (20221953)
- Maria Miguel Fonseca (20250380)

**GitHub repository: https://github.com/barbara-sousa-franco/MLOps_Project**

<font color='#2f94d7' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Set Up & Import Libraries](#1)
- [2. Load Data](#2)
- [3. Remove duplicates](#3)
- [4. Remove impossible values](#4)


# <font color='#2f94d7' size=6>**1. Set Up & Import Libraries**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import TargetEncoder, StandardScaler
from scipy.stats import chi2_contingency

In [ ]:
# global variables
SEED = 23
TEST_SIZE = 0.2

NON_RESIDENTIAL_TYPES = [
    'Land', 'Garage', 'Warehouse', 'Storage', 'Industrial', 'Commercial'
]
 
ENERGY_ORDER = ['A+', 'A', 'B', 'B-', 'C', 'D', 'E', 'F', 'G', 'No Rating']
 
PERCENTILE_THRESHOLDS = {
    'Price':             0.995,
    'LivingArea':        0.995,
    'TotalArea':         0.995,
    'LotSize':           0.995,
    'GrossArea':         0.995,
    'BuiltArea':         0.995,
    'NumberOfBedrooms':  0.995,
    'NumberOfBathrooms': 0.995,
    'NumberOfWC':        0.995,
    'TotalRooms':        0.990,
}

# <font color='#2f94d7' size=6>**2. Load Data**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

In [ ]:
df = pd.read_csv('../data/01_raw/portugal_listings.csv')
print(f"Raw data shape: {df.shape}")

# <font color='#2f94d7' size=6>**3. Pre-Split cleaning**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**3.1 Remove duplicates**</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

In [ ]:
duplicated_rows = df[df.duplicated()]
print(f"Number of duplicated rows: {len(duplicated_rows)}")
print(f"Percentage of duplicated rows: {len(duplicated_rows) / len(df) * 100:.2f}%")
df.drop_duplicates(keep=False,inplace=True)
print(f"Number of duplicated rows after dropping: {len(df[df.duplicated()])}")

# <font color='#2f94d7' size=6>**3.2 Remove columns**</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

In [ ]:
cols_to_drop = [
    'PublishDate',           # 78% nulls, 0.3% parseable
    'Floor',                 # 79% nulls
    'ConservationStatus',    # 86% nulls
    'EnergyEfficiencyLevel', # duplicate of EnergyCertificate
    'City',                  # cardinality 275, use District instead
    'Town',                  # cardinality 2263, use District instead
]
df = df.drop(columns=cols_to_drop)
print(f"Columns after drop: {df.shape[1]}")

# <font color='#2f94d7' size=6>**3.3 Remove rows outside of Portugal**</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)

In [ ]:
n_before = len(df)
df = df[df['District'] != 'Z - Fora de Portugal']
print(f"Rows removed (outside Portugal): {n_before - len(df)}")

# <font color='#2f94d7' size=6>**3.4 Convert impossible and improbable values to NA**</font> <a class="anchor" id="3_4"></a>

[Back to TOC](#toc)

In [ ]:
# Negative or zero values for certain features are not plausible in the context of real estate 
# listings. For example, a property cannot have a negative price or a negative area.
#  Similarly, a property cannot have zero bedrooms if it is listed as a residential property.
#  Therefore, we will identify and handle these implausible values by replacing them with NaN
#  (Not a Number) to indicate that they are missing or invalid. The date also has to be within 
# a reasonable range, as properties cannot be built in the far past or future.

impossible = {
    'Price':            df['Price'] <= 0,
    'GrossArea':        df['GrossArea'] < 0,
    'TotalArea':        df['TotalArea'] < 0,
    'LivingArea':       df['LivingArea'] < 0,
    'BuiltArea':        df['BuiltArea'] < 0,
    'LotSize':          df['LotSize'] < 0,
    'NumberOfWC':       df['NumberOfWC'] < 0,
    'NumberOfBathrooms':df['NumberOfBathrooms'] < 0,
    'NumberOfBedrooms': df['NumberOfBedrooms'] < 0,
    'TotalRooms':       df['TotalRooms'] < 0,
    'Parking':          df['Parking'] < 0,
    'ConstructionYear': (df['ConstructionYear'] > 2026) | (df['ConstructionYear'] < 1800),
}
for col, mask in impossible.items():
    df.loc[mask, col] = np.nan

In [ ]:
# In addition to impossible values, there are also implausible values that, while not strictly 
# impossible, are highly unlikely to be accurate. For example, a property listed with a price of
#  1 euro or a property with more than 20 bedrooms is likely to be an error in the data.
#  We will also handle these implausible values by replacing them with NaN.

implausible = {
    'Price':             df['Price'] == 1,
    'NumberOfBedrooms':  df['NumberOfBedrooms'] > 20,
    'NumberOfBathrooms': df['NumberOfBathrooms'] > 20,
    'NumberOfWC':        df['NumberOfWC'] > 20,
    'TotalRooms':        df['TotalRooms'] > 50,
}
for col, mask in implausible.items():
    df.loc[mask, col] = np.nan

# <font color='#2f94d7' size=6>**3.5 Remove rows without Price**</font> <a class="anchor" id="3_5"></a>

[Back to TOC](#toc)

In [ ]:
n_before = len(df)
df = df.dropna(subset=['Price'])
print(f"Rows removed (missing Price): {n_before - len(df)}")

# <font color='#2f94d7' size=6>**3.6 Consolidate EnergyCertificate**</font> <a class="anchor" id="3_6"></a>

[Back to TOC](#toc)

In [ ]:
no_rating_values = ['NC', 'Not available', 'No Certificate']
df['EnergyCertificate'] = df['EnergyCertificate'].replace(no_rating_values, 'No Rating')
df['EnergyCertificate'] = df['EnergyCertificate'].fillna('No Rating')

# <font color='#2f94d7' size=6>**3.7 Non-Residencial types consistency**</font> <a class="anchor" id="3_7"></a>

[Back to TOC](#toc)

In [ ]:
non_res_mask = df['Type'].isin(NON_RESIDENTIAL_TYPES)
zero_cols = [
    'NumberOfBedrooms', 'NumberOfBathrooms', 'NumberOfWC',
    'TotalRooms', 'LivingArea', 'BuiltArea'
]
for col in zero_cols:
    df.loc[non_res_mask, col] = df.loc[non_res_mask, col].fillna(0)

# <font color='#2f94d7' size=6>**3.8 Typecasting**</font> <a class="anchor" id="3.8"></a>

[Back to TOC](#toc)

In [ ]:
# Convert discrete columns to integer type
discrete_cols = [
    'NumberOfBedrooms', 'NumberOfBathrooms', 'NumberOfWC',
    'TotalRooms', 'Parking', 'ConstructionYear'
]
for col in discrete_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

In [ ]:
# Convert boolean columns to integer type (True/False to 1/0)
bool_cols = ['Garage', 'Elevator', 'ElectricCarsCharging', 'HasParking']
bool_map = {True: 1, False: 0, 'True': 1, 'False': 0}
for col in bool_cols:
    df[col] = df[col].map(bool_map)

# <font color='#2f94d7' size=6>**3.9 Ordinal Encoding for EnergyCertificate**</font> <a class="anchor" id="3.9"></a>

[Back to TOC](#toc)

In [ ]:
energy_map = {v: i for i, v in enumerate(reversed(ENERGY_ORDER))}
# A+ = 9 (best), No Rating = 0
df['EnergyCertificate'] = df['EnergyCertificate'].map(energy_map)

# <font color='#2f94d7' size=6>**3.10 Log1p(Price) as target**</font> <a class="anchor" id="3.9"></a>

[Back to TOC](#toc)

In [ ]:
df['Price_log'] = np.log1p(df['Price'])
 
print(f"\nShape after pre-split cleaning: {df.shape}")
print(f"Missing values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# <font color='#2f94d7' size=6>**3.11 Save to data/02_intermediate**</font> <a class="anchor" id="3.9"></a>

[Back to TOC](#toc)

In [ ]:
# Save the cleaned dataset to 02_intermediate
df.to_csv('../data/02_intermediate/portugal_listings_cleaned.csv', index=False)

# <font color='#2f94d7' size=6>**4. Train/Test Split**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

In [ ]:
y = df['Price_log']
X = df.drop(columns=['Price_log', 'Price'])
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED
)
 
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# <font color='#2f94d7' size=6>**4.1 Save to data/03_primary**</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

In [ ]:
X_train.to_csv('../data/03_primary/X_train.csv', index=False)
X_test.to_csv('../data/03_primary/X_test.csv', index=False)
y_train.to_csv('../data/03_primary/y_train.csv', index=False)
y_test.to_csv('../data/03_primary/y_test.csv', index=False)
print("Saved train/test splits to 03_primary/")